# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display main summary from metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")


## 2. Data Overview

Review the available record sets, their fields, and associated `@id` values from the Croissant schema. `mlcroissant` provides utilities to inspect the available entities.

In [ ]:
# List all available record sets and their field @id's
print("Available record sets and fields:")
record_sets = dataset.record_sets
for record_set in record_sets:
    print(f"RecordSet: {record_set['@id']} (name: {record_set.get('name', 'N/A')})")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):  # normalize to list
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            field_id = field['@id']
            field_name = field.get('name', '')
        else:
            field_id = field
            field_name = ''
        print(f"  - Field: {field_id} {f'(name: {field_name})' if field_name else ''}")


## 3. Data Extraction

Load the data from a specific record set using its `@id`. In this example, we will extract records from all available record sets. Please ensure you use the record sets and field `@id`s from the previous overview.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print('Record set @ids:', record_set_ids)
dataframes = {}

# Load each record set as a DataFrame
for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    if records:
        dataframes[rid] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from {rid}.")
    else:
        print(f"No records found for {rid}.")

# Pick the main record set for further analysis (typically the core data table)
if len(dataframes) == 0:
    raise Exception("No record sets with data found in this schema.")

# We'll use the first data-containing record set id for demonstration
main_record_set_id = list(dataframes.keys())[0]
print(f'Field names in {main_record_set_id}:', dataframes[main_record_set_id].columns.tolist())
# Preview data
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common preprocessing steps such as filtering, normalization, and grouping.

_Note: Replace `<numeric_field_id>` and `<group_field_id>` with actual `@id`s as appropriate for your dataset._

In [ ]:
from pandas.api.types import is_numeric_dtype

# List potential numeric fields from the DataFrame
df = dataframes[main_record_set_id]
numeric_candidates = [col for col in df.columns if is_numeric_dtype(df[col])]

# If no numeric columns, try to coerce candidates
if not numeric_candidates:
    float_candidates = []
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if is_numeric_dtype(df[col]):
                float_candidates.append(col)
        except Exception:
            pass
    numeric_candidates = float_candidates

if numeric_candidates:
    numeric_field = numeric_candidates[0] # Use the first numeric field for demonstration
    print(f"Selected numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to find a grouping field (categorical)
    cat_candidates = [col for col in df.columns if (df[col].dtype == 'object' and len(df[col].unique()) < len(df) // 2)]
    group_field = cat_candidates[0] if cat_candidates else None

    if group_field:
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization

Visualize numeric data distributions or relationships.

_This example creates a histogram for the selected numeric field, with optional grouping by a categorical field if present._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use previous variable context
if 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8,5))
    if 'group_field' in locals() and group_field:
        sns.histplot(data=df, x=numeric_field, hue=group_field, multiple='stack', kde=True)
        plt.title(f"Distribution of {numeric_field} grouped by {group_field}")
    else:
        sns.histplot(df[numeric_field], kde=True)
        plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

- Successfully loaded and explored the FAIR² dataset using the Croissant schema and `mlcroissant` tools.
- Identified available record sets and fields using the `@id` approach for robust referencing.
- Performed basic EDA and visualization on extracted data.
- This workflow is adaptable: use `@id`s for any other fields and record sets of interest, extend the EDA framework, and build domain-specific analyses.